# Stage 03 · Step 2 — Held-out test evaluation

Score the trained PPO policy on **test printers (85–99)** under the **same** `scalar_objective` used by Stages 01 & 02. Re-evaluate Stages 01 and 02 on the same printer set so the comparison is apples-to-apples.

Outputs:
- `results/per_printer_tau_test.csv` — Stage 03's per-printer τ + KPIs.
- `results/per_printer_tau_test_stage01.csv`, `..._stage02.csv` — same shape, with the constant τ replicated.
- `results/kpi_comparison.csv` — fleet-level KPI table for all three stages.
- `results/best_tau_per_printer.yaml` — the per-printer τ vectors as YAML.

In [ ]:
from __future__ import annotations
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

from ml_models import PROJECT_ROOT
from ml_models.lib.data import TEST_PRINTERS
from ml_models.lib.env_runner import default_dates
from ml_models.lib.objective import INFEASIBLE_FLOOR
from ml_models.lib.rl import (
    evaluate_constant_tau,
    evaluate_per_printer_policy,
    kpi_comparison_table,
    load_ppo,
    load_ssl_encoder,
    per_printer_table_for_constant_tau,
)
from sdg.generate import load_configs
from sdg.schema import COMPONENT_IDS

STAGE_DIR = PROJECT_ROOT / 'ml_models/03_rl+ssl'
MODELS_DIR = STAGE_DIR / 'models'
RESULTS_DIR = STAGE_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

STAGE_01_BEST = PROJECT_ROOT / 'ml_models/01_baseline/results/best_tau.yaml'
STAGE_02_BEST = PROJECT_ROOT / 'ml_models/02_ssl/results/best_tau_surrogate.yaml'

DATES = default_dates()
components_cfg, couplings_cfg, cities_cfg = load_configs()

## Stage 03: per-printer τ on test printers

In [ ]:
model = load_ppo(MODELS_DIR / 'ppo_policy_best.zip')
encoder = load_ssl_encoder()

stage03_per_printer, stage03_kpis = evaluate_per_printer_policy(
    model,
    printer_ids=list(TEST_PRINTERS),
    encoder_bundle=encoder,
    dates=DATES,
    components_cfg=components_cfg,
    couplings_cfg=couplings_cfg,
    cities_cfg=cities_cfg,
)
stage03_per_printer.to_csv(RESULTS_DIR / 'per_printer_tau_test.csv', index=False)

print('Stage 03 fleet KPIs:')
for k, v in stage03_kpis.items():
    print(f'  {k}: {v}')
stage03_per_printer.round(2)

## Stages 01 & 02: constant τ replicated across test printers

In [ ]:
stage01_per_printer = None
stage01_kpis = None
if STAGE_01_BEST.exists():
    with STAGE_01_BEST.open() as handle:
        stage01_payload = yaml.safe_load(handle)
    stage01_tau = {c: float(stage01_payload['tau_nom_h'][c]) for c in COMPONENT_IDS}
    stage01_per_printer = per_printer_table_for_constant_tau(
        stage01_tau,
        printer_ids=list(TEST_PRINTERS),
        dates=DATES,
        components_cfg=components_cfg,
        couplings_cfg=couplings_cfg,
        cities_cfg=cities_cfg,
    )
    stage01_per_printer.to_csv(RESULTS_DIR / 'per_printer_tau_test_stage01.csv', index=False)
    stage01_kpis = evaluate_constant_tau(
        stage01_tau,
        printer_ids=list(TEST_PRINTERS),
        dates=DATES,
        components_cfg=components_cfg,
        couplings_cfg=couplings_cfg,
        cities_cfg=cities_cfg,
    )
    print('Stage 01 fleet KPIs:')
    for k, v in stage01_kpis.items():
        print(f'  {k}: {v}')
else:
    print(f'{STAGE_01_BEST} not found; skipping Stage 01 eval')

In [ ]:
stage02_per_printer = None
stage02_kpis = None
if STAGE_02_BEST.exists():
    with STAGE_02_BEST.open() as handle:
        stage02_payload = yaml.safe_load(handle)
    stage02_tau = {c: float(stage02_payload['tau_nom_h'][c]) for c in COMPONENT_IDS}
    stage02_per_printer = per_printer_table_for_constant_tau(
        stage02_tau,
        printer_ids=list(TEST_PRINTERS),
        dates=DATES,
        components_cfg=components_cfg,
        couplings_cfg=couplings_cfg,
        cities_cfg=cities_cfg,
    )
    stage02_per_printer.to_csv(RESULTS_DIR / 'per_printer_tau_test_stage02.csv', index=False)
    stage02_kpis = evaluate_constant_tau(
        stage02_tau,
        printer_ids=list(TEST_PRINTERS),
        dates=DATES,
        components_cfg=components_cfg,
        couplings_cfg=couplings_cfg,
        cities_cfg=cities_cfg,
    )
    print('Stage 02 fleet KPIs:')
    for k, v in stage02_kpis.items():
        print(f'  {k}: {v}')
else:
    print(f'{STAGE_02_BEST} not found; skipping Stage 02 eval')

## Comparison table

Same `scalar_objective`, same printer set, same horizon — three policies.

In [ ]:
stage_definitions = []
per_printer_dfs = {'stage_03': stage03_per_printer}
fleet_kpis = {'stage_03': stage03_kpis}

if stage01_kpis is not None:
    stage_definitions.append(('stage_01', stage01_tau, 'Optuna over constant τ'))
    per_printer_dfs['stage_01'] = stage01_per_printer
    fleet_kpis['stage_01'] = stage01_kpis
if stage02_kpis is not None:
    stage_definitions.append(('stage_02', stage02_tau, 'SSL+RUL surrogate over constant τ'))
    per_printer_dfs['stage_02'] = stage02_per_printer
    fleet_kpis['stage_02'] = stage02_kpis
stage_definitions.append(('stage_03', None, 'PPO + frozen SSL encoder, per-printer τ'))

comparison = kpi_comparison_table(
    test_printers=list(TEST_PRINTERS),
    stage_definitions=stage_definitions,
    per_printer_dfs=per_printer_dfs,
    fleet_kpis=fleet_kpis,
)
comparison.to_csv(RESULTS_DIR / 'kpi_comparison.csv', index=False)
comparison

In [ ]:
best_tau_per_printer = {
    int(row['printer_id']): {c: float(row[f'tau_{c}']) for c in COMPONENT_IDS}
    for _, row in stage03_per_printer.iterrows()
}
payload = {
    'evaluated_on': 'test printers (id 85..99)',
    'fleet_value': float(stage03_kpis['value']),
    'fleet_annual_cost_eur_per_printer_year': float(stage03_kpis['annual_cost']),
    'fleet_availability': float(stage03_kpis['availability']),
    'fleet_deficit': float(stage03_kpis['deficit']),
    'tau_per_printer': best_tau_per_printer,
}
with (RESULTS_DIR / 'best_tau_per_printer.yaml').open('w', encoding='utf-8') as handle:
    yaml.safe_dump(payload, handle, sort_keys=False)
print('Wrote best_tau_per_printer.yaml')
print(yaml.safe_dump(
    {k: v for k, v in payload.items() if k != 'tau_per_printer'},
    sort_keys=False,
))

## Done criteria

- [ ] `kpi_comparison.csv` shows `stage_03.fleet_value < stage_02.fleet_value`.
- [ ] `stage_03.fleet_availability >= 0.95` (no infeasibility floor breach).
- [ ] Per-printer τ heatmap (next notebook) shows actual variation across printers — otherwise the policy collapsed to a constant and Stage 03 reduces to Stage 02.